# ขั้นที่ 3: Business Impact — การพยากรณ์ที่แม่นขึ้นมีค่าเป็นเงินเท่าไร

## ภาพรวมของขั้นนี้

| | |
|---|---|
| **คำถาม** | ถ้าใช้โมเดลจากขั้นที่ 2 แทนวิธีง่ายๆ จะลดความเสียหายได้กี่ดอลลาร์ |
| **ทำไมต้องรู้** | WRMSSE ที่ดีขึ้นยังไม่ใช่ภาษาของผู้บริหาร ต้องแปลงเป็นตัวเงินเพื่อใช้ตัดสินใจลงทุน |
| **ข้อมูล** | ผลพยากรณ์ของ backtest รอบล่าสุด (28 วันที่รู้ยอดจริงแล้ว) จากขั้นที่ 2 |

## แนวคิด: จำลองการสั่งของ

```
พยากรณ์ ──► ตั้งสต็อก = พยากรณ์ + safety stock ──► เทียบกับยอดขายจริง
                                                    ├─ ของไม่พอ → เสียยอดขาย (lost sales)   → เสียกำไร
                                                    └─ ของเกิน  → ค่าถือครอง + ลดราคาระบาย/ทิ้ง
```

- **Safety stock** = z × √พยากรณ์ (สต็อกกันเหนียวตามความไม่แน่นอน สมมติว่ายอดขายกระจายแบบ Poisson)
- ใช้ **กติกาเดียวกัน** กับทั้งโมเดลและ baseline ความต่างของต้นทุนจึงมาจากความแม่นของพยากรณ์เท่านั้น

## ขั้นตอน

```
3.1 สมมติฐาน  →  3.2 ภาพใหญ่: ต้นทุนรวม  →  3.3 Drill down: รายหมวด  →  3.4 Drill down: สินค้ารายตัว
            →  3.5 ขยายผลเป็นรายปีและทุกสาขา  →  3.6 Sensitivity  →  3.7 สรุป
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True

FIGURES = ROOT / 'reports' / 'figures'
ANALYSIS = ROOT / 'analysis'
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)


def show(fig, name):
    """แสดงกราฟในหน้านี้ และบันทึกสำเนาไว้ใน reports/figures"""
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

from src.data import PROCESSED

---
## 3.1 สมมติฐาน

ข้อมูล M5 ไม่มีต้นทุนสินค้า ระดับสต็อก และข้อมูลของขาดจริง จึงต้องตั้งสมมติฐาน ตัวเลขทั้งหมดในขั้นนี้ขึ้นกับค่าเหล่านี้
หัวข้อ 3.6 จะทดสอบว่าผลเปลี่ยนแค่ไหนเมื่อสมมติฐานเปลี่ยน

| สมมติฐาน | ค่า | เหตุผล |
|---|---|---|
| Service level (z) | 1.28 | สต็อกพอขายประมาณ 90% ของวัน |
| กำไรขั้นต้น | 25% | ระดับทั่วไปของค้าปลีก |
| ต้นทุนถือครองสต็อก | 25% ต่อปี | ระดับทั่วไปของค้าปลีก (ดอกเบี้ย, พื้นที่, ประกัน) |
| Markdown / waste | 5% ของมูลค่าของเกิน | ส่วนที่ต้องลดราคาระบายหรือทิ้ง |
| จำนวนสาขาที่ขยายผล | 4,600 | ประมาณจำนวนสาขา Walmart ในสหรัฐฯ (ใช้เพื่อแสดงขนาดของโอกาสเท่านั้น) |

In [ ]:
ASSUMPTIONS = {
    'service_z': 1.28,
    'gross_margin': 0.25,
    'carrying_cost_annual': 0.25,
    'excess_loss_rate': 0.05,
    'stores_total': 4_600,
}
STORES_IN_DATA = 10
a = ASSUMPTIONS

bt = np.load(PROCESSED / 'backtest_last_fold.npz', allow_pickle=True)
y_true, price, cat_id = bt['y_true'], np.nan_to_num(bt['price']), bt['cat_id']
base_name, model_name = f"Baseline ({bt['baseline']})", f"Model ({bt['best']})"
forecasts = {base_name: bt['pred_baseline'], model_name: bt['pred_best']}
print(f"ช่วงที่จำลอง: d_{int(bt['end']) + 1}–d_{int(bt['end']) + 28} | เทียบ {base_name} กับ {model_name}")

In [ ]:
def stock_level(pred, z):
    return np.ceil(pred + z * np.sqrt(pred))


def simulate(pred, z=a['service_z'], margin=a['gross_margin'], carrying_cost=a['carrying_cost_annual'],
             excess_loss_rate=a['excess_loss_rate']):
    """ต้นทุนจากการพยากรณ์ผิดใน 28 วัน แยกตามหมวด (ดอลลาร์)"""
    stock = stock_level(pred, z)
    lost = np.maximum(y_true - stock, 0) * price
    excess = np.maximum(stock - y_true, 0) * price
    out = pd.DataFrame({'cat_id': cat_id, 'lost_sales': lost.sum(axis=1), 'excess_inventory': excess.sum(axis=1)})
    out = out.groupby('cat_id').sum()
    out.loc['ALL'] = out.sum()
    out['lost_margin'] = out['lost_sales'] * margin
    out['carrying_cost'] = out['excess_inventory'] * carrying_cost / 365
    out['markdown_waste'] = out['excess_inventory'] * excess_loss_rate
    out['total_cost'] = out[['lost_margin', 'carrying_cost', 'markdown_waste']].sum(axis=1)
    return out


impact = pd.concat({name: simulate(p) for name, p in forecasts.items()}, names=['method'])
COST_PARTS = ['lost_margin', 'carrying_cost', 'markdown_waste']

---
## 3.2 ภาพใหญ่: ต้นทุนรวมจากการพยากรณ์ผิด (10 สาขา, 28 วัน)

In [ ]:
total = impact.xs('ALL', level='cat_id')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
total[COST_PARTS].plot.bar(ax=axes[0], stacked=True, rot=0, color=['indianred', 'steelblue', 'darkorange'])
for i, v in enumerate(total['total_cost']):
    axes[0].text(i, v * 1.02, f'${v:,.0f}', ha='center', fontweight='bold')
axes[0].set(title='Total cost of forecast error (28 days, 10 stores)', xlabel='', ylabel='$')

total[['lost_sales', 'excess_inventory']].plot.bar(ax=axes[1], rot=0, color=['indianred', 'steelblue'])
axes[1].set(title='Value of units short vs units in excess', xlabel='', ylabel='$')
show(fig, '03_2_total_cost')
total.T

---
## 3.3 Drill down: แยกตามหมวดสินค้า

ผลประหยัด = ต้นทุนของ baseline − ต้นทุนของโมเดล (ค่าบวกแปลว่าโมเดลดีกว่า)

In [ ]:
saving = impact.loc[base_name] - impact.loc[model_name]

fig, ax = plt.subplots(figsize=(9, 4.5))
saving.drop('ALL')[COST_PARTS].plot.bar(ax=ax, stacked=True, rot=0, color=['indianred', 'steelblue', 'darkorange'])
ax.axhline(0, color='black', lw=0.8)
for i, v in enumerate(saving.drop('ALL')['total_cost']):
    ax.text(i, max(v, 0) * 1.03, f'${v:,.0f}', ha='center', fontweight='bold')
ax.set(title='Saving vs baseline by category (28 days, 10 stores)', xlabel='', ylabel='$ saved')
show(fig, '03_3_saving_by_category')
saving[['lost_margin', 'carrying_cost', 'markdown_waste', 'total_cost']]

---
## 3.4 Drill down: สินค้ารายตัว

ดูสินค้าที่ทำรายได้สูงสุดว่าแต่ละวันสต็อกที่ตั้งตามพยากรณ์ของแต่ละวิธี เทียบกับยอดขายจริงเป็นอย่างไร
- พื้นที่ **สีแดง** = ของไม่พอขาย, พื้นที่ **สีฟ้า** = ของเกิน

In [ ]:
top = int(np.argmax((y_true * price).sum(axis=1)))
days = np.arange(1, 29)

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, (name, pred) in zip(axes, forecasts.items()):
    stock = stock_level(pred[top], a['service_z'])
    ax.step(days, stock, where='mid', color='black', label='Stock level')
    ax.bar(days, y_true[top], color='lightgray', label='Actual sales')
    ax.fill_between(days, stock, y_true[top], where=y_true[top] > stock, step='mid', color='indianred', alpha=0.6, label='Short')
    ax.fill_between(days, stock, y_true[top], where=y_true[top] < stock, step='mid', color='steelblue', alpha=0.3, label='Excess')
    ax.set(title=f"{name}: {bt['item_id'][top]} @ {bt['store_id'][top]}", xlabel='Day of forecast', ylabel='Units')
axes[0].legend(fontsize=8)
show(fig, '03_4_item_example')

---
## 3.5 ขยายผล: ต่อปี และทั้งองค์กร

- **ต่อปี:** ผล 28 วัน × 365/28
- **ทุกสาขา:** ผลของ 10 สาขา × (4,600 / 10) เป็นการประมาณขนาดของโอกาสอย่างหยาบ เพราะสาขาอื่นอาจมีขนาดและสินค้าต่างจาก 10 สาขานี้

In [ ]:
scale = pd.DataFrame({
    '28 days, 10 stores': saving['total_cost'],
    'per year, 10 stores': saving['total_cost'] * 365 / 28,
    'per year, all stores (rough)': saving['total_cost'] * 365 / 28 * a['stores_total'] / STORES_IN_DATA,
})
scale.to_csv(ANALYSIS / 'business_impact.csv')
scale.style.format('${:,.0f}')

---
## 3.6 Sensitivity: ผลยังเป็นบวกไหมถ้าสมมติฐานเปลี่ยน

ผลประหยัดต่อปีใน 10 สาขา เมื่อเปลี่ยน **กำไรขั้นต้น** (มีผลต่อมูลค่าของขาด) และ **markdown/waste** (มีผลต่อมูลค่าของเกิน)
สีเขียว = โมเดลคุ้มกว่า, สีแดง = baseline คุ้มกว่า

In [ ]:
LOSS_RATES, MARGINS = [0.02, 0.05, 0.10, 0.20], [0.15, 0.25, 0.35]
grid = pd.DataFrame(index=[f'{x:.0%}' for x in LOSS_RATES], columns=[f'{m:.0%}' for m in MARGINS], dtype=float)
for loss in LOSS_RATES:
    for margin in MARGINS:
        costs = {n: simulate(p, margin=margin, excess_loss_rate=loss).loc['ALL', 'total_cost'] for n, p in forecasts.items()}
        grid.loc[f'{loss:.0%}', f'{margin:.0%}'] = (costs[base_name] - costs[model_name]) * 365 / 28

fig, ax = plt.subplots(figsize=(8, 4.5))
lim = np.abs(grid.values).max()
sns.heatmap(grid / 1e6, annot=True, fmt='.2f', cmap='RdYlGn', center=0, vmin=-lim / 1e6, vmax=lim / 1e6, ax=ax,
            cbar_kws={'label': '$ million per year'})
ax.set(title='Annual saving, 10 stores ($ million)', xlabel='Gross margin', ylabel='Markdown / waste rate')
show(fig, '03_6_sensitivity')

---
## 3.7 สรุปขั้นที่ 3

SUMMARY_PLACEHOLDER

**ข้อจำกัดของการจำลอง**
- เป็นการจำลองแบบวันต่อวัน ไม่ได้คิดสต็อกที่เหลือข้ามวัน รอบการสั่งของ และ lead time จริง
- ข้อมูลยอดขายจริงอาจต่ำกว่าความต้องการจริงในวันที่ของหมด (censored demand) ทำให้ lost sales จริงอาจสูงกว่านี้
- ต้องยืนยันผลด้วยการทดลองจริง (pilot) ตามข้อเสนอในขั้นที่ 4